In [ ]:
import utils
print(f"utils : {utils.__version__}")
from utils.structure import build_electrode, build_nanoribbon, build_reduced_device, find_nearest_atoms, infer_centersize
from utils.plots import plot_with_center, mark_electrode # plotting
from utils.energies import hamiltonian, multi_LDOS_parallel, multi_LDOS
from utils.loader import path_finder, load_datastructure
from utils import pretty_print_columns
from utils._wrappers import timeit


import sisl
import numpy as np
import matplotlib.pyplot as plt
from ase.visualize import view
from tqdm.notebook import tqdm

import ipywidgets
from ipywidgets import interact
from fractions import Fraction


%reload_ext line_profiler


# Global parameters

In [ ]:
ENERGIES = np.linspace(-2, 2, num=50)

# Benchmark full device: flake convergence
This is how I expect the final structure to converge when computing the LDOS from self-energies. Calculated using [flake_size_conv.py](../../QTM/flake_size_conv.py)

<img src="../../QTM/figures/flake_size_conv.svg">

In [ ]:
@timeit
def electrode_dos(ham, nk, energies):
    """compute DOS from built-ins
    
    Parameters
    ----------
    ham : Hamiltonian of shape (K, K)
        Hamiltonian in question 
    nk : int
        number of k points
    energies : ndarray of shape (N,)
        list of energies to compute the dos

    Returns
    -------
    dos : ndarray of shape (N,)
        density of state
    bs : ndarray of shape (M, N)
        bandstructure eigenvalues 
    lk : ndarray of shape (M,)
        scaled kpoints to linear values for use in plotting
    """
    dist = sisl.get_distribution(method="lorentzian", smearing=0.05)
    
    bz = sisl.MonkhorstPack(ham, [nk, 1, 1])
    kvecs = [[0,0,0], [0.5, 0, 0]]
    bands = sisl.BandStructure(ham, kvecs, 15, [r"$\Gamma$", "$K$"])
    lk, kticks, knames = bands.lineark(ticks=True)
    bs = bands.apply.array.eigh()
    dos = (bz.apply.average).eigenstate(wrap = lambda x: x.DOS(energies, distribution=dist)) 
    return dos, bs, (lk, kticks, knames)

In [ ]:
def plot_dos_band(device, nk, energies, LWC):
    L, W, C = LWC
    """Plot the device band structure and DOS"""
    H = hamiltonian(device)
    dos, bands, k = electrode_dos(H, nk, energies)
    fig, axes = plt.subplots(1,2, sharey=True)
    fig.suptitle(f"Built-in calc L={L}, W={W}, C={C}")
    axes[0].set_title("Band structure")    
    axes[0].plot(k[0], bands)    
    axes[0].set_xlabel("$k$", size=15)
    axes[0].set_xticks(k[1])
    axes[0].set_xticklabels(k[2])
    
    axes[1].set_title("DOS")
    axes[1].plot(dos, energies, label="built-in DOS")
    axes[1].set_xlabel("DOS", size=15)
    
    axes[0].set_ylabel("E", size=15)
    fig.tight_layout()
    for ax in axes:
        ax.grid()
    
    return fig, axes

# Benchmark Electrode: self-energy method against built-in

In [ ]:
i = 1
length = 2
lenght, width, center_size = infer_centersize(length, i)

__params__ribbon = f"{length}L_{width}W_{center_size}C"
print(f"{__params__ribbon = }")
electrode = build_electrode(width=width, length=length)
ribbon = build_nanoribbon(electrode=electrode, center_size=center_size)
device_ribbon, lr_idx_ribbon = build_reduced_device(ribbon, electrode, repeat=1)
# device_ribbon.write(f"structures/ribbon_{__params__ribbon}.xyz")
atoms_style = mark_electrode(lr_idx_ribbon)
device_ribbon.plot(axes="xy", atoms_style=atoms_style)

plot_with_center(device_ribbon, atoms_style=atoms_style)

## Built-in calculations for bands and DOS

In [ ]:
fig, axes = plot_dos_band(electrode, 100, ENERGIES, (lenght, width, center_size))
minmax = 2
axes[0].set(ylim=(-minmax, minmax))
fig.savefig(f"figures/electrode_bands_DOS_{__params__ribbon}.png")

## Self-energy LDOS VS. built-in DOS 

# Convergence for device: Reduced structure (3 arms)

In [ ]:
i = 1
length = 2

length, width, center_size = infer_centersize(length, i)
# width += 2
__params__electrode = f"{length}L_{width}W"
__params__device = __params__electrode + f"_{center_size}C"
electrode = build_electrode(width=width, length=length)
ribbon = build_nanoribbon(electrode=electrode, center_size=center_size)
device, lr_idx_device = build_reduced_device(ribbon, electrode, repeat=3)
print(f"{__params__device.split("_")}")
atoms_style = mark_electrode(lr_idx_device)

plot_with_center(device, atoms_style=atoms_style)

In [ ]:
l = 1
i = 4
LWC = infer_centersize(l, i)
# l, w, c = 1, 37, 18
electrode, ribbon, device, DATA = load_datastructure(LWC=LWC)
energies, ldos, lr_idx = DATA["energies"], DATA["ldos"], DATA["lr_idx"]

Nk, NE, Na = ldos.shape # number of k-points, energy values, and atoms for device
atoms_style = mark_electrode(lr_idx)



plot_with_center(device, atoms_style=atoms_style)

## Plotting LDOS VS. DOS*
*DOS is likely not right - calculated assuming 'pristine'

In [ ]:
def interactive_params(device, Nk):
    # Determine k-point options
    DEVICE_CENTER_ATOMS = find_nearest_atoms(device.xyz, device.center(), neighbours=6)
    kpts = sisl.MonkhorstPack(hamiltonian(device), [Nk, 1, 1])
    options = [(f"{[str(Fraction(val).limit_denominator(len(kpts)*2)) for val in kvec]})", i) for i, kvec in enumerate(kpts)]
    ks = ipywidgets.Dropdown(options=options, description="k")
    
    # Determine site (only the centermost)
    sites = ipywidgets.Dropdown(options=DEVICE_CENTER_ATOMS, description="Site")
    
    # Option for using normalized data : (val - mean) / std
    norm = ipywidgets.Checkbox(value=False, description="Normalize", indent=True)
    # same = ipywidgets.Checkbox(value=True, description="Plot in same figure", indent=True)
    return ks, sites, norm


In [ ]:
from sisl.viz.processors.math import normalize
def plot_ldos(kidx, s, norm=True):
    """Plot LDOS from self-energy calculation."""
    # print(ldos.shape)
    # print(f"{sites.options = }")
    vmin = ldos[kidx, :, sites.options].mean()
    # print(f"{vmin}")
    
    X1 = ldos[kidx, :, s]
    
    fig, axes = plt.subplots(1,1)
    if norm: # plot in the save figure
        X1 = normalize(X1, vmin=vmin, vmax = 1)
        axes.set_xlabel("LDOS (normalized)")
    else:
        axes.set_xlabel(f"LDOS")
    
    axes.plot(X1, energies, label="SE (LDOS)")
    axes.set_ylabel("E")
    axes.axvline(0)
    
    
    # title_strlist = PARAMS.split("_") # split string to list with "_" as seperator
    # parse_number = lambda string: int("".join(ch for ch in string if ch.isdigit())) # find number in string
    # for title_str in title_strlist:
    #     if "W" in title_str: # find width
    #         w = parse_number(title_str)
    #     elif "L" in title_str: # find length
    #         l = parse_number(title_str)
    #     elif "C" in title_str: # find center size
    #         c = parse_number(title_str)
    fig.suptitle("L={:}, W={:}, C={:}".format(*LWC))

ks, sites, norm = interactive_params(device, Nk=Nk)
interact(plot_ldos, kidx=ks, s=sites, norm=norm)

## Plot LDOS pr. energy (interactive)

In [ ]:
from sisl.viz.processors.math import normalize
def scale_by_ldos(ldos, Eidx, atoms_style = None):
    scaling = lambda E: 10*E + 0.02
    Nk, NE, Ns = ldos.shape
    C_ATOMS = find_nearest_atoms(device.xyz, device.center(), neighbours=6)
    vmin = ldos[..., C_ATOMS].mean()
    
    
    norm_ldos = normalize(ldos, vmin=vmin, vmax=1)
    style = {"atoms": range(Ns),
             "size": scaling(norm_ldos[0, Eidx, :])}
    if atoms_style is None:
        atoms_style = []
    
    atoms_style.append(style)
    return atoms_style
    
style = scale_by_ldos(ldos, 0, atoms_style)

In [ ]:
print(PARAMS)
_atoms_style = mark_electrode(lr_idx)

def proj_ldos(Eidx):
    _style = scale_by_ldos(ldos, Eidx, _atoms_style)
    fig = device.plot(axes="xy", atoms_style=_style, backend="matplotlib", show_cell=False)
        
        
    fig.suptitle(f"{PARAMS}, E = {ENERGIES[Eidx]:.3f}")
    fig.show()
    # return fig

# idxs = ipywidgets.Dropdown(options=range(ENERGIES.shape[0]), description="E")
idxs = ipywidgets.IntSlider(min=0, max=len(ENERGIES)-1, description="E")
interact(proj_ldos, Eidx=idxs)

# Note to self:
* Adjust SE-calc: $\Sigma = i\mathrm{LDOS}(E=0)c$, in Hamiltonian for the LDOS calculation find $c$
    * Try different cancstants.
    * Goal: reduced edge states when plotting PDOS (using LDOS) around E=0
* Plot LDOS (as currently) together with electrode DOS to verify behavior. Try for different widths
* Plot BG Vs. electrode width to find correlation (hopefully converging). 
    * BG is in this case equal to "flat-region" around E=0 in the LDOS plot
* Make new routine for computing LDOS for only center atoms (don't solve for all sites if used for plotting LDOS in center region).
    * Make identical routine, but only solving for atom index for the center region.


**Update:**  
See [electrode_dos](notebooks/electrode_DOS.ipynb) for attempt at combining built-in DOS in *y* and self-energy in *x*
